# Tumor CM network outputs

This notebook generates only the tumor-centric CM nodeplot, its legends/colorbars, and the tumor Top10 node-correlation heatmap.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D


BASE_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_analysis")
OUTPUT_DIR = BASE_DIR / "balanced_joint_nmf_outputs"
JOINT_CM_DIR = OUTPUT_DIR / "joint_cm"
NETWORK_DIR = JOINT_CM_DIR / "networks"
TABLE_DIR = JOINT_CM_DIR / "tables"
SHARED_DIR = OUTPUT_DIR / "shared"
FIGURE_DIR = JOINT_CM_DIR / "figures"

EDGE_TABLE = NETWORK_DIR / "status_specific_nodeplot_edges.csv"
NODE_TABLE = NETWORK_DIR / "tumor_network_nodes_from_H_df.csv"
TOP10_NODE_TABLE = TABLE_DIR / "joint_cm_cell_subtype_nodes_top10_from_H_df.csv"
NORM_FREQUENCY_TABLE = SHARED_DIR / "non_epi_subtype_frequency_global_minmax.csv"
SAMPLE_STATUS_TABLE = SHARED_DIR / "sample_status.csv"

OUT_STEM = "tumor_centric_nodeplot_edge_origin"
EDGE_THRESHOLD = 0.25

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


def prefix_color_map_from_nodes(nodes: list[str]) -> dict[str, tuple[float, float, float]]:
    prefixes = sorted({node.split("_")[0] for node in nodes})
    palette = plt.get_cmap("tab20").colors
    return {prefix: palette[i % len(palette)] for i, prefix in enumerate(prefixes)}


def save_colorbar(cmap, norm, path_stem: Path, label: str) -> None:
    fig, ax = plt.subplots(figsize=(4.0, 0.45))
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=ax, orientation="horizontal")
    cb.set_label(label, fontsize=10)
    cb.ax.tick_params(labelsize=8)
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(path_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)


def save_edge_class_legend(path_stem: Path) -> None:
    handles = [
        Line2D([0], [0], color="#1d4ed8", lw=3, label="Tumor only"),
        Line2D([0], [0], color="#b91c1c", lw=3, label="Shared with normal-like"),
    ]
    fig, ax = plt.subplots(figsize=(3.8, 1.1))
    ax.axis("off")
    ax.legend(handles=handles, loc="center", frameon=False, ncol=2)
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(path_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)


def top10_nodes_by_cm() -> dict[str, list[str]]:
    top10 = pd.read_csv(TOP10_NODE_TABLE)
    top10 = top10.sort_values(["CM", "rank"])
    return top10.groupby("CM", sort=False)["node"].apply(list).to_dict()


def plot_status_node_correlation_heatmap(
    context_label: str,
    samples: pd.Index,
    norm_df: pd.DataFrame,
    top10_nodes: dict[str, list[str]],
    output_stem: Path,
) -> None:
    cm_names = list(top10_nodes)
    n_cols = 3
    n_rows = int(np.ceil(len(cm_names) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.2, n_rows * 4.15), squeeze=False)
    axes = axes.flatten()

    image = None
    for ax, cm_name in zip(axes, cm_names):
        nodes = [node for node in top10_nodes[cm_name] if node in norm_df.columns]
        ax.set_title(cm_name, fontsize=11, fontweight="bold")
        if len(nodes) < 2 or len(samples) < 4:
            ax.axis("off")
            ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center", fontsize=9)
            continue

        corr = norm_df.loc[samples, nodes].corr(method="pearson").loc[nodes, nodes]
        image = ax.imshow(corr.to_numpy(dtype=float), cmap="RdBu_r", vmin=-1.0, vmax=1.0, aspect="equal")
        ax.set_xticks(np.arange(len(nodes)))
        ax.set_yticks(np.arange(len(nodes)))
        ax.set_xticklabels(nodes, rotation=90, ha="center", fontsize=6)
        ax.set_yticklabels(nodes, fontsize=6)
        ax.tick_params(length=0)
        for spine in ax.spines.values():
            spine.set_visible(False)

    for ax in axes[len(cm_names):]:
        ax.axis("off")

    if image is not None:
        cb = fig.colorbar(image, ax=axes.tolist(), shrink=0.55, pad=0.015)
        cb.set_label("Pearson correlation (r)", fontsize=10)
        cb.ax.tick_params(labelsize=8)

    fig.suptitle(
        f"{context_label} CM top10 node-node correlation heatmaps",
        y=1.01,
        fontweight="bold",
        fontsize=15,
    )
    fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight", dpi=300)
    fig.savefig(output_stem.with_suffix(".svg"), bbox_inches="tight", dpi=300)
    plt.close(fig)




def edge_key(node_a: str, node_b: str) -> tuple[str, str]:
    return tuple(sorted((node_a, node_b)))


def plot_tumor_outputs() -> None:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    edges = pd.read_csv(EDGE_TABLE)
    nodes = pd.read_csv(NODE_TABLE)
    nodes = nodes.sort_values(["CM", "rank"])
    norm_df = pd.read_csv(NORM_FREQUENCY_TABLE, index_col=0)
    sample_status = pd.read_csv(SAMPLE_STATUS_TABLE, index_col=0)
    top10_nodes = top10_nodes_by_cm()

    tumor_edges = edges.loc[edges["context"].eq("tumor")].copy()
    normal_edges = edges.loc[edges["context"].eq("normal-like")].copy()

    normal_pass = {
        (row.CM, edge_key(row.node_a, row.node_b)): bool(row["edge_pass_r_ge_0.25"])
        for _, row in normal_edges.iterrows()
    }

    cm_names = nodes["CM"].drop_duplicates().tolist()
    all_nodes = nodes["node"].tolist()
    prefix_colors = prefix_color_map_from_nodes(all_nodes)

    shared_cmap = LinearSegmentedColormap.from_list("shared_red", ["#fee2e2", "#b91c1c"])
    tumor_only_cmap = LinearSegmentedColormap.from_list("tumor_only_blue", ["#dbeafe", "#1d4ed8"])
    edge_norm = Normalize(vmin=EDGE_THRESHOLD, vmax=1.0)

    n_cols = 2
    n_rows = int(np.ceil(len(cm_names) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.4, n_rows * 4.2), squeeze=False)
    axes = axes.flatten()

    summary = []
    for ax, cm_name in zip(axes, cm_names):
        cm_nodes = nodes.loc[nodes["CM"].eq(cm_name), "node"].tolist()
        cm_tumor_edges = tumor_edges.loc[tumor_edges["CM"].eq(cm_name)].copy()
        cm_tumor_edges = cm_tumor_edges.loc[cm_tumor_edges["edge_pass_r_ge_0.25"].astype(bool)]

        graph = nx.Graph()
        graph.add_nodes_from(cm_nodes)

        shared_count = 0
        tumor_only_count = 0
        for _, row in cm_tumor_edges.iterrows():
            pair = edge_key(row.node_a, row.node_b)
            is_shared = normal_pass.get((cm_name, pair), False)
            edge_class = "shared" if is_shared else "tumor_only"
            if is_shared:
                shared_count += 1
            else:
                tumor_only_count += 1
            graph.add_edge(row.node_a, row.node_b, weight=float(row.pearson_r), edge_class=edge_class)

        summary.append({"CM": cm_name, "shared_edges": shared_count, "tumor_only_edges": tumor_only_count})

        ax.set_title(cm_name, fontsize=14, fontweight="bold")
        ax.axis("off")
        if not cm_nodes:
            ax.text(0.5, 0.5, "No retained nodes", ha="center", va="center", fontsize=10)
            continue

        pos = nx.circular_layout(cm_nodes)

        if graph.number_of_edges() > 0:
            for edge_class, cmap in [("shared", shared_cmap), ("tumor_only", tumor_only_cmap)]:
                class_edges = [(a, b, d) for a, b, d in graph.edges(data=True) if d["edge_class"] == edge_class]
                if not class_edges:
                    continue
                nx.draw_networkx_edges(
                    graph,
                    pos,
                    edgelist=[(a, b) for a, b, _ in class_edges],
                    edge_color=[cmap(edge_norm(d["weight"])) for _, _, d in class_edges],
                    width=[2.0 + 3.0 * edge_norm(d["weight"]) for _, _, d in class_edges],
                    alpha=0.88,
                    ax=ax,
                )

        node_colors = [prefix_colors.get(node.split("_")[0], "#999999") for node in cm_nodes]
        nx.draw_networkx_nodes(
            graph,
            pos,
            nodelist=cm_nodes,
            node_color=node_colors,
            node_size=1700,
            linewidths=0.8,
            edgecolors="white",
            ax=ax,
        )
        nx.draw_networkx_labels(
            graph,
            pos,
            labels={node: node for node in cm_nodes},
            font_size=8,
            font_color="black",
            ax=ax,
        )

    for ax in axes[len(cm_names):]:
        ax.axis("off")

    fig.suptitle(
        "Tumor-centric CM node plots: tumor-only vs shared edges",
        y=1.005,
        fontweight="bold",
        fontsize=16,
    )
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{OUT_STEM}.pdf", bbox_inches="tight", dpi=300)
    fig.savefig(FIGURE_DIR / f"{OUT_STEM}.svg", bbox_inches="tight", dpi=300)
    plt.close(fig)

    save_colorbar(
        tumor_only_cmap,
        edge_norm,
        FIGURE_DIR / f"{OUT_STEM}_tumor_only_edge_colorbar",
        "Tumor-only edge correlation in tumor samples (r)",
    )
    save_colorbar(
        shared_cmap,
        edge_norm,
        FIGURE_DIR / f"{OUT_STEM}_shared_edge_colorbar",
        "Shared edge correlation in tumor samples (r)",
    )
    save_edge_class_legend(FIGURE_DIR / f"{OUT_STEM}_edge_class_legend")

    tumor_samples = sample_status.index[sample_status["status"].eq("tumor")].intersection(norm_df.index)
    plot_status_node_correlation_heatmap(
        "Tumor",
        tumor_samples,
        norm_df,
        top10_nodes,
        FIGURE_DIR / "tumor_top10_node_correlation_heatmap_no_edge_filter",
    )

    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
    print("Total shared edges:", int(summary_df["shared_edges"].sum()))
    print("Total tumor-only edges:", int(summary_df["tumor_only_edges"].sum()))
    print("Saved:", FIGURE_DIR / f"{OUT_STEM}.pdf")
    print("Saved:", FIGURE_DIR / f"{OUT_STEM}.svg")
    print("Saved:", FIGURE_DIR / "tumor_top10_node_correlation_heatmap_no_edge_filter.pdf")

plot_tumor_outputs()
